# Traffic flow pipeline — Smart City Week, Problem 2

YOLO11 + ByteTrack → line-crossing counts + turning movements → CSVs for the dashboard.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. In Google Drive, create `MyDrive/smart_city/videos/` and upload the mentor videos there
3. Run cells top to bottom. Edit **CONFIG** (cell 3) for each video — use the coordinate-picker cell to read line endpoints off the real video resolution.

Outputs land in `MyDrive/smart_city/outputs/<video_name>/`:
`events.csv` (every counted crossing), `tracks_od.csv` (turning movements, if zones defined), `counts_by_bucket.csv`, `counts_by_line.csv`, `od_matrix.csv`, `annotated.mp4` (demo footage). Download that folder and point `dashboard.py` at it.

In [ ]:
!pip -q install ultralytics
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L   # should show a Tesla T4 — if not, fix the runtime type before continuing

## Config
Line coordinates below are placeholders scaled from a 980x504 screenshot — **replace them** with values read off the picker cell (next) at the video's real resolution.
Counting-line rules: anchor near the stop line / crosswalk of each leg, perpendicular to travel, outside the intersection box, never at frame edges.

In [ ]:
from pathlib import Path

DRIVE      = Path('/content/drive/MyDrive/smart_city')
VIDEO_PATH = DRIVE / 'videos' / 'intersection_01.mp4'      # <-- change per video
OUT_DIR    = DRIVE / 'outputs' / VIDEO_PATH.stem
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME       = 'yolo11s.pt'
CLASSES          = {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}   # COCO ids
CONF             = 0.25        # low-ish on purpose: dense scooters
FRAME_STRIDE     = 3           # process every Nth frame (3 -> ~10 fps effective)
CHECKPOINT_EVERY = 500         # processed frames between CSV flushes (Colab disconnect insurance)

# Counting lines: name -> ((x1, y1), (x2, y2)) in pixel coords of the REAL video
LINES = {
    'A_right_leg':     ((627, 237), (902, 186)),
    'B_left_exit':     ((167, 166), (59, 312)),
    'C_near_approach': ((314, 438), (686, 398)),
}

# Optional entry/exit zones for turning-movement (OD) counts.
# Draw one polygon per intersection leg, at the leg mouth (not deep inside the box).
# Example: 'right_leg': [(640, 260), (900, 170), (960, 210), (700, 320)]
ZONES = {}

ANNOTATE         = True
ANNOTATE_RANGE_S = None        # e.g. (240, 360) to only render 4:00-6:00 and speed things up
BUCKET_SECONDS   = 60
PCU = {'motorcycle': 0.3, 'car': 1.0, 'bus': 2.0, 'truck': 2.5}   # passenger-car-unit weights

## Coordinate picker — read line/zone endpoints off this grid

In [ ]:
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(str(VIDEO_PATH))
cap.set(cv2.CAP_PROP_POS_FRAMES, 300)
ok, frame = cap.read()
cap.release()
assert ok, f'could not read {VIDEO_PATH} - check the path'
h, w = frame.shape[:2]
print(f'video resolution: {w} x {h}')

plt.figure(figsize=(18, 10))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.xticks(range(0, w, 100), rotation=90)
plt.yticks(range(0, h, 100))
plt.grid(color='yellow', alpha=0.4)
plt.title('Read (x, y) endpoints for LINES / ZONES off this grid, update CONFIG above')
plt.show()

## Tracking + counting loop
Counts a track once per line, at the moment its bottom-center point crosses the line segment. Checkpoints CSVs to Drive every `CHECKPOINT_EVERY` frames, so a Colab disconnect costs you nothing.

In [ ]:
import csv
import time
import numpy as np
import cv2
from ultralytics import YOLO


def seg_intersect(p1, p2, q1, q2):
    """True if segment p1->p2 crosses segment q1->q2."""
    def ccw(a, b, c):
        return (c[1] - a[1]) * (b[0] - a[0]) > (b[1] - a[1]) * (c[0] - a[0])
    return ccw(p1, q1, q2) != ccw(p2, q1, q2) and ccw(p1, p2, q1) != ccw(p1, p2, q2)


def cross_direction(line, prev, cur):
    """+1 / -1 depending on which side-to-side the movement crossed the line."""
    (ax, ay), (bx, by) = line
    lx, ly = bx - ax, by - ay
    mx, my = cur[0] - prev[0], cur[1] - prev[1]
    return 1 if lx * my - ly * mx > 0 else -1


def point_in_zone(pt, poly):
    return cv2.pointPolygonTest(np.array(poly, np.int32), (float(pt[0]), float(pt[1])), False) >= 0


def mmss(t):
    return f'{int(t) // 60:02d}:{int(t) % 60:02d}'


model = YOLO(MODEL_NAME)
cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = cap.get(cv2.CAP_PROP_FPS) or 30
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f'{w}x{h} @ {fps:.1f} fps, {n_frames} frames ({n_frames / fps / 60:.1f} min)')

writer = None
if ANNOTATE:
    writer = cv2.VideoWriter(str(OUT_DIR / 'annotated.mp4'),
                             cv2.VideoWriter_fourcc(*'mp4v'),
                             max(fps / FRAME_STRIDE, 5), (w, h))

events = []                 # [t_seconds, video_time, line, direction, track_id, class]
last_pos = {}               # track_id -> previous anchor point
counted = set()             # (track_id, line_name) pairs already counted
tracks = {}                 # track_id -> {cls, first_zone, last_zone, first_t, last_t}
line_counts = {ln: {c: 0 for c in CLASSES.values()} for ln in LINES}


def flush():
    with open(OUT_DIR / 'events.csv', 'w', newline='') as f:
        wr = csv.writer(f)
        wr.writerow(['t_seconds', 'video_time', 'line', 'direction', 'track_id', 'class'])
        wr.writerows(events)
    with open(OUT_DIR / 'tracks_od.csv', 'w', newline='') as f:
        wr = csv.writer(f)
        wr.writerow(['track_id', 'class', 'origin', 'destination', 'first_t', 'last_t'])
        for tid, i in tracks.items():
            wr.writerow([tid, i['cls'], i['first_zone'], i['last_zone'],
                         round(i['first_t'], 2), round(i.get('last_t', 0), 2)])


frame_idx, processed, t0 = 0, 0, time.time()
while True:
    ok, frame = cap.read()
    if not ok:
        break
    if frame_idx % FRAME_STRIDE:
        frame_idx += 1
        continue
    t = frame_idx / fps
    r = model.track(frame, persist=True, classes=list(CLASSES), conf=CONF, verbose=False)[0]

    if r.boxes is not None and r.boxes.id is not None:
        for (x1, y1, x2, y2), tid, ci in zip(r.boxes.xyxy.tolist(),
                                             r.boxes.id.int().tolist(),
                                             r.boxes.cls.int().tolist()):
            cname = CLASSES[ci]
            anchor = ((x1 + x2) / 2, y2)          # bottom-center = ground contact point
            info = tracks.setdefault(tid, {'cls': cname, 'first_zone': None,
                                           'last_zone': None, 'first_t': t})
            info['last_t'] = t
            for zname, poly in ZONES.items():
                if point_in_zone(anchor, poly):
                    info['first_zone'] = info['first_zone'] or zname
                    info['last_zone'] = zname
            prev = last_pos.get(tid)
            if prev:
                for lname, line in LINES.items():
                    if (tid, lname) not in counted and seg_intersect(prev, anchor, *line):
                        counted.add((tid, lname))
                        d = cross_direction(line, prev, anchor)
                        events.append([round(t, 2), mmss(t), lname, d, tid, cname])
                        line_counts[lname][cname] += 1
            last_pos[tid] = anchor

    if writer is not None and (ANNOTATE_RANGE_S is None
                               or ANNOTATE_RANGE_S[0] <= t <= ANNOTATE_RANGE_S[1]):
        vis = r.plot(line_width=2)
        for lname, ((ax_, ay_), (bx_, by_)) in LINES.items():
            cv2.line(vis, (int(ax_), int(ay_)), (int(bx_), int(by_)), (0, 215, 255), 3)
            cv2.putText(vis, f'{lname}: {sum(line_counts[lname].values())}',
                        (int(min(ax_, bx_)), int(min(ay_, by_)) - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 215, 255), 2)
        for poly in ZONES.values():
            cv2.polylines(vis, [np.array(poly, np.int32)], True, (80, 220, 80), 2)
        writer.write(vis)

    processed += 1
    frame_idx += 1
    if processed % CHECKPOINT_EVERY == 0:
        flush()
        rate = processed / (time.time() - t0)
        totals = ', '.join(f'{k}={sum(v.values())}' for k, v in line_counts.items())
        print(f'{frame_idx}/{n_frames} frames | {mmss(t)} of video | {rate:.0f} fps | {totals}')

cap.release()
if writer is not None:
    writer.release()
flush()
print('done ->', OUT_DIR)

## Aggregate metrics (feeds the dashboard)

In [ ]:
import pandas as pd

ev = pd.read_csv(OUT_DIR / 'events.csv')
ev['bucket'] = (ev['t_seconds'] // BUCKET_SECONDS).astype(int) * BUCKET_SECONDS

by_bucket = ev.pivot_table(index='bucket', columns='class', values='track_id',
                           aggfunc='count').fillna(0).astype(int)
for c in PCU:
    if c not in by_bucket.columns:
        by_bucket[c] = 0
by_bucket['total'] = by_bucket[list(PCU)].sum(axis=1)
by_bucket['pcu'] = sum(by_bucket[c] * wt for c, wt in PCU.items())
by_bucket.to_csv(OUT_DIR / 'counts_by_bucket.csv')

win = max(1, 300 // BUCKET_SECONDS)
roll = by_bucket['total'].rolling(win).sum()
if roll.notna().any():
    end = int(roll.idxmax())
    print(f'busiest 5-min window ends at {end}s ({end // 60:02d}:{end % 60:02d}) '
          f'with {int(roll.max())} vehicles')

ev.groupby(['line', 'direction', 'class']).size().unstack(fill_value=0) \
  .to_csv(OUT_DIR / 'counts_by_line.csv')
print('vehicle mix:')
print(ev['class'].value_counts(normalize=True).round(3))

od = pd.read_csv(OUT_DIR / 'tracks_od.csv').dropna(subset=['origin', 'destination'])
od = od[od['origin'] != od['destination']]
if len(od):
    od_matrix = od.pivot_table(index='origin', columns='destination',
                               values='track_id', aggfunc='count').fillna(0).astype(int)
    od_matrix.to_csv(OUT_DIR / 'od_matrix.csv')
    print('turning movements:')
    print(od_matrix)
else:
    print('no OD data (define ZONES in CONFIG to get turning-movement counts)')

## Webster signal timing (step 03 — decision conversion)
Feed it the highest flow (PCU/hr) per signal phase. Scale a 5-min peak to hourly by multiplying by 12. State assumed values for approaches the camera doesn't see.

In [ ]:
def webster(critical_flows_pcu_hr, sat_flow=1800, lost_time_s=12):
    """Webster optimal cycle. Returns (cycle_s, green_splits_s, Y). None if oversaturated."""
    Y = sum(f / sat_flow for f in critical_flows_pcu_hr)
    if Y >= 0.95:
        return None, None, Y
    C = (1.5 * lost_time_s + 5) / (1 - Y)
    g_total = C - lost_time_s
    splits = [round(g_total * (f / sat_flow) / Y, 1) for f in critical_flows_pcu_hr]
    return round(C, 1), splits, round(Y, 3)

# example: measured approach peaks at 900 PCU/hr, assume 500 PCU/hr on the cross street
C, g, Y = webster([900, 500])
print(f'flow ratio Y={Y} -> recommended cycle ~ {C}s, green splits ~ {g}')

## Accuracy check (20 minutes of work, huge credibility)
Hand-count one 2-minute window in the raw video, then compare:

In [ ]:
t1, t2 = 120, 240   # the window you hand-counted
chk = ev[(ev['t_seconds'] >= t1) & (ev['t_seconds'] < t2)]
print(chk.groupby(['line', 'class']).size().unstack(fill_value=0))
print(f'total in window: {len(chk)} -> accuracy = ours / your hand count')

**Per-video workflow:** change `VIDEO_PATH` in CONFIG, redo the picker + lines for that camera, re-run the loop and aggregation. Each 20-min clip ≈ 4–6 min on a T4.